In [ ]:
# This cell is for all imports
import os, json, cv2, random
import matplotlib.pyplot as plt
import numpy as np

from detectron2 import model_zoo
from detectron2.data.datasets import register_coco_instances
from detectron2.utils.visualizer import Visualizer
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.config import get_cfg

In [ ]:
# This cell registers the datasets
register_coco_instances("train", {}, "train/annotations.json", "train/images")
register_coco_instances("test", {}, "test/annotations.json", "test/images")

DatasetCatalog.get("train")

In [ ]:
# This cell is to visualize some of the training data samples 
dataset_dicts = DatasetCatalog.get("train")
figure = plt.figure(figsize=(8, 8))
cols, rows = 3, 3
for i, d in enumerate(random.sample(dataset_dicts, cols * rows)):
    img = cv2.imread(d["file_name"])
    visualizer = Visualizer(img[:, :, ::-1], metadata=MetadataCatalog.get("train"), scale=0.5, font_size_scale=0.3)
    out = visualizer.draw_dataset_dict(d)
    
    figure.add_subplot(rows, cols, i+1)
    plt.axis("off")
    plt.imshow(out.get_image()[:, :, ::-1])

In [ ]:
# This cell prints the keys of the first image in the train dataset
print(DatasetCatalog.get("train")[0].keys())

In [ ]:
# This cell is for configuration and training of the model 
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
cfg.DATASETS.TRAIN = ("train",)
cfg.DATASETS.TEST = ()
cfg.DATALOADER.NUM_WORKERS = 2
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")  # Let training initialize from model zoo
cfg.SOLVER.IMS_PER_BATCH = 2  # This is the real "batch size" commonly known to deep learning people
cfg.SOLVER.BASE_LR = 0.00025  # pick a good LR
cfg.SOLVER.MAX_ITER = 3600 # 3768 total images, half as many iterations with a batch size of 2
cfg.SOLVER.STEPS = []        # do not decay learning rate
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128 # small for now, probably should be 512
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 2
# NOTE: this config means the number of classes, but a few popular unofficial tutorials incorrect uses num_classes+1 here.
cfg.OUTPUT_DIR = "./output"
cfg.MODEL.DEVICE='cpu'
cfg.DATALOADER.FILTER_EMPTY_ANNOTATIONS = True

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
trainer = DefaultTrainer(cfg) 
trainer.resume_or_load(resume=True)
trainer.train()

In [ ]:
# Inference should use the config with parameters that are used in training
# cfg now already contains everything we've set previously. We changed it a little bit for inference:
cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")  # path to the model we just trained
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.7   # set a custom testing threshold
predictor = DefaultPredictor(cfg)

In [ ]:
# This cell tests the trained model against human annotations
from detectron2.utils.visualizer import ColorMode
dataset_dicts = DatasetCatalog.get("test")
figure = plt.figure(figsize=(8, 8))
cols, rows = 2, 3
for i, d in enumerate(random.sample(dataset_dicts, rows)):    
    im = cv2.imread(d["file_name"])
    outputs = predictor(im)  # format is documented at https://detectron2.readthedocs.io/tutorials/models.html#model-output-format
    v = Visualizer(im[:, :, ::-1],
                   metadata=MetadataCatalog.get("test"), 
                   scale=0.5, 
                   instance_mode=ColorMode.IMAGE_BW,   # remove the colors of unsegmented pixels. This option is only available for segmentation models
                   font_size_scale=0
    )
    out = v.draw_instance_predictions(outputs["instances"].to("cpu"))
    figure.add_subplot(rows, cols, 2*i+1)
    plt.axis("off")
    plt.imshow(out.get_image()[:, :, ::-1])
    
    v = Visualizer(im[:, :, ::-1],
                   metadata=MetadataCatalog.get("test"), 
                   scale=0.5, 
                   instance_mode=ColorMode.IMAGE_BW   # remove the colors of unsegmented pixels. This option is only available for segmentation models
    )

    actual = None
    for ann in d['annotations']:
        for segm in ann['segmentation']:
            actual = v.draw_polygon(np.reshape(segm, (int(len(segm)/2), 2)), "tab:green")
    
    figure.add_subplot(rows, cols, 2*i+2)
    plt.axis("off")
    if actual is not None:
        plt.imshow(actual.get_image()[:, :, ::-1])
    else:
        plt.imshow(im)

In [ ]:
# This cell returns the standard evaluation metrics for the model
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

evaluator = COCOEvaluator("test", output_dir="./detectron_output")
val_loader = build_detection_test_loader(cfg, "test")
print(inference_on_dataset(predictor.model, val_loader, evaluator))